# 19 correct da binomial glm

**Purpose:** Recalculate the binomial-GLM sensitivity analysis with corrected dtypes.

**Original analysis notebook:** `18b_fix_DA_binomial_GLM_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 37b — Fix binomial GLM sensitivity test for differential abundance

This notebook only fixes the binomial GLM sensitivity output from the DA/composition step.  
The primary DA results remain the sample-level Kruskal-Wallis and Mann-Whitney tests.

Input:
`DA_sample_level_compartment_proportions_HVG8000_r0_6.tsv`

Outputs:
- `DA_glm_binomial_compartment_sample_level_HVG8000_r0_6_v2_fixed.tsv`
- `DA_glm_binomial_compartment_sample_level_HVG8000_r0_6_v2_fixed_coefficients.tsv`
- `DA_glm_binomial_fix_manifest_HVG8000_r0_6.tsv`


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import chi2

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_TSV = METADATA_DIR / "DA_sample_level_compartment_proportions_HVG8000_r0_6.tsv"

OUT_GLM = METADATA_DIR / "DA_glm_binomial_compartment_sample_level_HVG8000_r0_6_v2_fixed.tsv"
OUT_COEF = METADATA_DIR / "DA_glm_binomial_compartment_sample_level_HVG8000_r0_6_v2_fixed_coefficients.tsv"
OUT_MANIFEST = METADATA_DIR / "DA_glm_binomial_fix_manifest_HVG8000_r0_6.tsv"

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


In [ ]:

df = pd.read_csv(INPUT_TSV, sep="\t")

# Keep final compartment-level sample summaries only
df = df.loc[df["feature_type"].astype(str).eq("final_compartment_major")].copy()

# Force numeric dtypes. This fixes the old GLM dtype error.
for col in ["n_cells_feature", "n_cells_total", "proportion"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["condition"] = df["condition"].astype(str)
df["feature"] = df["feature"].astype(str)
df["unit_id"] = df["unit_id"].astype(str)

# Remove invalid rows
df = df.dropna(subset=["n_cells_feature", "n_cells_total", "proportion", "condition"])
df = df.loc[df["n_cells_total"] > 0].copy()

print(df.shape)
display(df.head())
display(df.groupby("condition")["unit_id"].nunique())


In [ ]:

# Binomial GLM sensitivity analysis:
# success = n cells in compartment
# total = total cells in sample
# model: proportion ~ condition
#
# This is a sensitivity check, not the main DA test.
# Main inference should still prioritize sample-level non-parametric tests.

import statsmodels.api as sm
import patsy

glm_rows = []
coef_rows = []

for feature, sub in df.groupby("feature", sort=True):
    sub = sub.copy()

    # Need at least 2 conditions with data
    cond_counts = sub["condition"].value_counts()
    valid_conditions = cond_counts.index.tolist()
    if len(valid_conditions) < 2:
        glm_rows.append({
            "feature": feature,
            "test": "binomial_GLM_condition_LRT_v2",
            "pvalue": np.nan,
            "status": "skipped:<2_conditions",
            "n_units": int(sub["unit_id"].nunique()),
            "n_conditions": int(len(valid_conditions)),
            "ll_full": np.nan,
            "ll_null": np.nan,
            "df_diff": np.nan,
            "lrt_stat": np.nan,
        })
        continue

    # Set GBM as reference if present for more interpretable coefficients
    cond_order = ["GBM", "LGG", "ndGBM", "rGBM"]
    present_order = [c for c in cond_order if c in set(sub["condition"])]
    extra = [c for c in sorted(set(sub["condition"])) if c not in present_order]
    cat_order = present_order + extra
    sub["condition"] = pd.Categorical(sub["condition"], categories=cat_order, ordered=False)

    try:
        # Endog as two-column success/failure is robust for binomial counts
        successes = sub["n_cells_feature"].astype(float).values
        failures = (sub["n_cells_total"] - sub["n_cells_feature"]).astype(float).values
        y = np.column_stack([successes, failures])

        X_full = patsy.dmatrix("C(condition)", sub, return_type="dataframe")
        X_null = patsy.dmatrix("1", sub, return_type="dataframe")

        full_model = sm.GLM(y, X_full, family=sm.families.Binomial())
        null_model = sm.GLM(y, X_null, family=sm.families.Binomial())

        full_res = full_model.fit(maxiter=200)
        null_res = null_model.fit(maxiter=200)

        lrt_stat = 2 * (full_res.llf - null_res.llf)
        df_diff = int(full_res.df_model - null_res.df_model)
        pvalue = float(chi2.sf(lrt_stat, df_diff)) if df_diff > 0 else np.nan

        glm_rows.append({
            "feature": feature,
            "test": "binomial_GLM_condition_LRT_v2",
            "pvalue": pvalue,
            "status": "ok",
            "n_units": int(sub["unit_id"].nunique()),
            "n_conditions": int(len(valid_conditions)),
            "ll_full": float(full_res.llf),
            "ll_null": float(null_res.llf),
            "df_diff": df_diff,
            "lrt_stat": float(lrt_stat),
        })

        for term, coef, se, z, p in zip(
            full_res.params.index,
            full_res.params.values,
            full_res.bse.values,
            full_res.tvalues.values,
            full_res.pvalues.values,
        ):
            coef_rows.append({
                "feature": feature,
                "term": term,
                "coef_log_odds": float(coef),
                "odds_ratio": float(np.exp(coef)),
                "std_error": float(se),
                "z_value": float(z),
                "pvalue": float(p),
                "status": "ok",
                "reference_condition": cat_order[0] if cat_order else None,
            })

    except Exception as e:
        glm_rows.append({
            "feature": feature,
            "test": "binomial_GLM_condition_LRT_v2",
            "pvalue": np.nan,
            "status": f"error:{repr(e)}",
            "n_units": int(sub["unit_id"].nunique()),
            "n_conditions": int(len(valid_conditions)),
            "ll_full": np.nan,
            "ll_null": np.nan,
            "df_diff": np.nan,
            "lrt_stat": np.nan,
        })

glm_df = pd.DataFrame(glm_rows)
coef_df = pd.DataFrame(coef_rows)

# FDR correction
try:
    from statsmodels.stats.multitest import multipletests
    ok = glm_df["pvalue"].notna()
    glm_df["qvalue_fdr_bh"] = np.nan
    if ok.sum() > 0:
        glm_df.loc[ok, "qvalue_fdr_bh"] = multipletests(glm_df.loc[ok, "pvalue"], method="fdr_bh")[1]

    okc = coef_df["pvalue"].notna() if len(coef_df) else pd.Series([], dtype=bool)
    if len(coef_df):
        coef_df["qvalue_fdr_bh"] = np.nan
        coef_df.loc[okc, "qvalue_fdr_bh"] = multipletests(coef_df.loc[okc, "pvalue"], method="fdr_bh")[1]
except Exception as e:
    print("[WARNING] FDR correction failed:", repr(e))

glm_df = glm_df.sort_values(["qvalue_fdr_bh", "pvalue"], na_position="last")
coef_df = coef_df.sort_values(["feature", "term"]) if len(coef_df) else coef_df

write_tsv_replace(glm_df, OUT_GLM)
write_tsv_replace(coef_df, OUT_COEF)

display(glm_df)
display(coef_df.head(20))


In [ ]:

manifest = pd.DataFrame([
    {"path": str(INPUT_TSV), "exists": INPUT_TSV.exists(), "size_bytes": INPUT_TSV.stat().st_size if INPUT_TSV.exists() else np.nan},
    {"path": str(OUT_GLM), "exists": OUT_GLM.exists(), "size_bytes": OUT_GLM.stat().st_size if OUT_GLM.exists() else np.nan},
    {"path": str(OUT_COEF), "exists": OUT_COEF.exists(), "size_bytes": OUT_COEF.stat().st_size if OUT_COEF.exists() else np.nan},
])
write_tsv_replace(manifest, OUT_MANIFEST)
display(manifest)
